In [5]:
# Setup: create project folder and download dataset
import os
import subprocess

project_path = os.path.expanduser(r"~\Desktop\retail_project")
os.makedirs(project_path, exist_ok=True)
os.chdir(project_path)

result = subprocess.run(
    ["python", "-m", "kaggle", "datasets", "download", "-d",
     "mohammadtalib786/retail-sales-dataset", "--unzip"],
    capture_output=True, text=True
)
print(result.stdout)

print("Files:", os.listdir("."))

Dataset URL: https://www.kaggle.com/datasets/mohammadtalib786/retail-sales-dataset
License(s): CC0-1.0


Files: ['retail_sales_dataset.csv']


In [6]:
import pandas as pd
df = pd.read_csv("retail_sales_dataset.csv")

print("Shape:", df.shape)
print(df.head())
print("Columns:", df.columns.tolist())
print(df.dtypes)

Shape: (1000, 9)
   Transaction ID        Date Customer ID  Gender  Age Product Category  \
0               1  2023-11-24     CUST001    Male   34           Beauty   
1               2  2023-02-27     CUST002  Female   26         Clothing   
2               3  2023-01-13     CUST003    Male   50      Electronics   
3               4  2023-05-21     CUST004    Male   37         Clothing   
4               5  2023-05-06     CUST005    Male   30           Beauty   

   Quantity  Price per Unit  Total Amount  
0         3              50           150  
1         2             500          1000  
2         1              30            30  
3         1             500           500  
4         2              50           100  
Columns: ['Transaction ID', 'Date', 'Customer ID', 'Gender', 'Age', 'Product Category', 'Quantity', 'Price per Unit', 'Total Amount']
Transaction ID      int64
Date                  str
Customer ID           str
Gender                str
Age                 int64
Prod

Each row in this dataset represents a single sales transaction at a retail store. It contains information about the customer (Customer ID, Gender, Age), the product sold (Product Category, Quantity, Price per Unit), the total amount of the transaction, and the date the transaction occurred.

## Milestone 2: Python Basics, OOP, and Exceptions

In [3]:
# Take the first 20 rows and convert to a list of dictionaries
first_20 = df.head(20).to_dict('records')
print(first_20[0])  # show one example row

{'Transaction ID': 1, 'Date': '2023-11-24', 'Customer ID': 'CUST001', 'Gender': 'Male', 'Age': 34, 'Product Category': 'Beauty', 'Quantity': 3, 'Price per Unit': 50, 'Total Amount': 150}


In [4]:
# List comprehension: get customer IDs of transactions with quantity > 2
big_orders = [row['Customer ID'] for row in first_20 if row['Quantity'] > 2]
print(big_orders)

['CUST001', 'CUST008', 'CUST010', 'CUST012', 'CUST013', 'CUST014', 'CUST015', 'CUST016', 'CUST017', 'CUST020']


In [5]:
# map + lambda: calculate total price for each transaction (Quantity * Price per Unit)
totals = list(map(lambda r: r['Quantity'] * r['Price per Unit'], first_20))
print(totals)

# filter + lambda: keep only transactions where total > 100
high_value = list(filter(lambda r: r['Quantity'] * r['Price per Unit'] > 100, first_20))
print(len(high_value), "transactions with total > 100")

[150, 1000, 30, 500, 100, 30, 50, 100, 600, 200, 100, 75, 1500, 120, 2000, 1500, 100, 50, 50, 900]
10 transactions with total > 100


In [6]:
# Sort transactions by Total Amount, highest first
sorted_rows = sorted(first_20, key=lambda r: r['Total Amount'], reverse=True)

# print just the Customer ID and Total Amount for clarity
for r in sorted_rows[:5]:
    print(r['Customer ID'], '-', r['Total Amount'])

CUST015 - 2000
CUST013 - 1500
CUST016 - 1500
CUST002 - 1000
CUST020 - 900


In [7]:
# Custom function: calculate average order value from a list of transactions
def average_order_value(rows):
    total = sum(r['Total Amount'] for r in rows)
    return total / len(rows) if rows else 0

avg = average_order_value(first_20)
print("Average order value (first 20 rows):", avg)

Average order value (first 20 rows): 457.75


In [ ]:
# Custom exception for invalid transactions
class InvalidTransactionError(Exception):
    """Raised when a transaction has invalid price, quantity, or category"""
    pass
class SaleTransaction:
    def __init__(self, customer_id, age, quantity, price_per_unit):
        # Encapsulation: use underscore prefix to protect internal data
        self._customer_id = customer_id
        self._age = age
        self._quantity = quantity
        self._price_per_unit = price_per_unit
        self._validate()

    def _validate(self):
        # Reject invalid transactions
        if self._quantity <= 0 or self._price_per_unit <= 0:
            raise InvalidTransactionError(
                f"Invalid quantity or price for customer {self._customer_id}"
            )

    def calculate_total(self):
        # Basic calculation method (will be overridden by child classes)
        return self._quantity * self._price_per_unit

    def __repr__(self):
        return f"{self.__class__.__name__}({self._customer_id}, total={self.calculate_total()})"


In [13]:
class MaleSale(SaleTransaction):
    def calculate_total(self):
        # Example polymorphism: could add different logic here later
        return super().calculate_total()


class FemaleSale(SaleTransaction):
    def calculate_total(self):
        # Example polymorphism: apply a small 5% loyalty discount
        return super().calculate_total() * 0.95
    # Create real transaction objects from the first 20 rows
transactions = []

for _, row in df.head(20).iterrows():
    try:
        if row['Gender'] == 'Male':
            t = MaleSale(row['Customer ID'], row['Age'], row['Quantity'], row['Price per Unit'])
        else:
            t = FemaleSale(row['Customer ID'], row['Age'], row['Quantity'], row['Price per Unit'])
        transactions.append(t)
    except InvalidTransactionError as e:
        print("Skipped invalid row:", e)

# print all created objects
for t in transactions:
    print(t)

MaleSale(CUST001, total=150)
FemaleSale(CUST002, total=950.0)
MaleSale(CUST003, total=30)
MaleSale(CUST004, total=500)
MaleSale(CUST005, total=100)
FemaleSale(CUST006, total=28.5)
MaleSale(CUST007, total=50)
MaleSale(CUST008, total=100)
MaleSale(CUST009, total=600)
FemaleSale(CUST010, total=190.0)
MaleSale(CUST011, total=100)
MaleSale(CUST012, total=75)
MaleSale(CUST013, total=1500)
MaleSale(CUST014, total=120)
FemaleSale(CUST015, total=1900.0)
MaleSale(CUST016, total=1500)
FemaleSale(CUST017, total=95.0)
FemaleSale(CUST018, total=47.5)
FemaleSale(CUST019, total=47.5)
MaleSale(CUST020, total=900)


## Milestone 3: NumPy Processing

In [10]:
import numpy as np

# Convert two numeric columns to NumPy arrays
quantity_arr = df['Quantity'].to_numpy()
total_arr = df['Total Amount'].to_numpy()
print(quantity_arr[:10])   # show first 10 values
print(total_arr[:10])
# Vectorized operation: apply a 10% price increase to Total Amount
total_with_increase = total_arr * 1.10
print(total_with_increase[:10])
# Vectorized operation: apply a 10% price increase to Total Amount
total_with_increase = total_arr * 1.10
print(total_with_increase[:10])
# Boolean filtering: keep only totals greater than 500
high_totals = total_arr[total_arr > 500]
print(high_totals)
print("Count of high-value transactions:", len(high_totals))
# Statistics on one numeric column (Total Amount)
mean_val = np.mean(total_arr)
max_val = np.max(total_arr)
min_val = np.min(total_arr)
std_val = np.std(total_arr)

print("Mean:", mean_val)
print("Max:", max_val)
print("Min:", min_val)
print("Standard Deviation:", std_val)



[3 2 1 1 2 1 2 4 2 4]
[ 150 1000   30  500  100   30   50  100  600  200]
[ 165. 1100.   33.  550.  110.   33.   55.  110.  660.  220.]
[ 165. 1100.   33.  550.  110.   33.   55.  110.  660.  220.]
[1000  600 1500 2000 1500  900 1000  900 1200  900  900  900 1200 1500
  900 1000 1500  900 1200 2000 1200 2000 2000 1500 2000 2000 1000  600
 1000 1200  600 1000 1200 2000  900 1500 1500 1500 1000 2000 2000  600
  600  900  600 1000 2000 1200 1500 2000 1000  900 2000 2000  600 1000
 1500 1200 2000 1500  900  900 1200  900 1500  900  900 1500 1200 1000
 1500 1500 1500 1500  600  600 1000 1000 1500  900  900 2000 1000 2000
  900  900 2000 1000 1000 1500 2000  600  600 1200  900 1200 1000 1200
 1000  600 1200 1500 1200  900 1200 1200  900 1200 2000 1000  600 1000
 1500 1200 1200 1500 1500  600  600 1000 1500  600  600 1000 1000  600
  600  600 1000 1200  900  900 2000 2000  900 1000 2000 1500 1200  900
 1200 1000  900 1200  600 1200  600 2000 1500 1000  900 1200 1000 1200
 1500  600 1000  900 

## Milestone 4: Pandas Analysis and Excel Export

In [13]:
# Check for missing values in each column
print(df.isnull().sum())

Transaction ID      0
Date                0
Customer ID         0
Gender              0
Age                 0
Product Category    0
Quantity            0
Price per Unit      0
Total Amount        0
dtype: int64


In [15]:
# Check for duplicate rows
print("Number of duplicate rows:", df.duplicated().sum())

# Remove duplicates if any
df = df.drop_duplicates()
print("Shape after removing duplicates:", df.shape)
# Convert Date column to actual date type
df['Date'] = pd.to_datetime(df['Date'])
print(df['Date'].dtype)
print(df['Date'].head())

Number of duplicate rows: 0
Shape after removing duplicates: (1000, 9)
datetime64[us]
0   2023-11-24
1   2023-02-27
2   2023-01-13
3   2023-05-21
4   2023-05-06
Name: Date, dtype: datetime64[us]


In [18]:
# Q1: Total spending by gender
spending_by_gender = df.groupby('Gender')['Total Amount'].sum()
print(spending_by_gender)
# Q2: Total quantity sold per product category
quantity_by_category = df.groupby('Product Category')['Quantity'].sum().sort_values(ascending=False)
print(quantity_by_category)
# Q3: Average customer age by gender
avg_age_by_gender = df.groupby('Gender')['Age'].mean()
print(avg_age_by_gender)

Gender
Female    232840
Male      223160
Name: Total Amount, dtype: int64
Product Category
Clothing       894
Electronics    849
Beauty         771
Name: Quantity, dtype: int64
Gender
Female    41.356863
Male      41.428571
Name: Age, dtype: float64


In [19]:
# Pivot table: total sales amount by Gender and Product Category
pivot = pd.pivot_table(
    df,
    values='Total Amount',
    index='Gender',
    columns='Product Category',
    aggfunc='sum'
)
print(pivot)


Product Category  Beauty  Clothing  Electronics
Gender                                         
Female             74830     81275        76735
Male               68685     74305        80170


In [21]:
%pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpy


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
with pd.ExcelWriter('retail_sales_report.xlsx') as writer:
    df.describe().to_excel(writer, sheet_name='Summary Stats')
    spending_by_gender.to_excel(writer, sheet_name='Spending by Gender')
    quantity_by_category.to_excel(writer, sheet_name='Quantity by Category')
    avg_age_by_gender.to_excel(writer, sheet_name='Avg Age by Gender')
    pivot.to_excel(writer, sheet_name='Pivot Table')

print("Excel file created successfully: retail_sales_report.xlsx")

Excel file created successfully: retail_sales_report.xlsx
